# Inventario multisensor – Sismo de Venezuela (Mw 7.2 + 7.5, 24/06/2026)
# *Multi-sensor inventory – Venezuela earthquake doublet (24 June 2026)*
**Sentinel-1 (C) · NISAR (L) · Umbra (X) · Planet (óptico / optical)**

🇪🇸 Ejecute las celdas **en orden, de arriba hacia abajo**. Si Colab se reinicia, vuelva a correr desde la **Celda 1**. Los resultados se guardan en Google Drive (`InSAR_LaGuaira_Resultados/inventario_2026`).

🇬🇧 Run the cells **in order, top to bottom**. If Colab restarts, start again from **Cell 1**. Outputs are written to Google Drive (`InSAR_LaGuaira_Resultados/inventario_2026`).

## Celda 1 · Instalar librerías / *Cell 1 · Install libraries* (≈1–2 min)

In [ ]:
!pip install -q asf_search hyp3_sdk rasterio geopandas shapely contextily pyarrow duckdb boto3 pystac-client requests
print('Librerías instaladas')


## Celda 2 · Google Drive y carpeta de trabajo / *Cell 2 · Google Drive and working folder*
🇪🇸 Todo se escribe dentro de Drive, así no se pierde si se cae la sesión.  
🇬🇧 Everything is written to Drive so nothing is lost if the session drops.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
CARPETA = '/content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026'
os.makedirs(CARPETA, exist_ok=True)
os.chdir(CARPETA)
print('Trabajando en:', os.getcwd())


## Celda 3 · Credenciales NASA Earthdata / *Cell 3 · NASA Earthdata credentials*
🇪🇸 Solo se necesitan para **descargar** y para HyP3. Las búsquedas funcionan sin ellas.  
🇬🇧 Only needed to **download** and for HyP3. Searches work without them.

In [ ]:
import asf_search as asf
import getpass
usuario = input('Usuario Earthdata: ')
clave = getpass.getpass('Contraseña: ')
session = asf.ASFSession().auth_with_creds(usuario, clave)
print('Autenticación correcta')


## Celda 4 · Inventario Umbra (banda X) / *Cell 4 · Umbra inventory (X-band)*
🇪🇸 Lee las 26 adquisiciones del bucket público y sus metadatos STAC.  
🇬🇧 Reads the 26 collects in the public bucket and their STAC metadata.

In [ ]:
import json
import os
from collections import defaultdict
from itertools import combinations

import boto3
import numpy as np
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "umbra-open-data-catalog"
PREFIJO = "sar-data/tasks/Venezuela_Earthquake_Support/"
SALIDA = "./umbra_inventario"
os.makedirs(SALIDA, exist_ok=True)

# Umbrales para considerar dos adquisiciones "misma geometría" (detección de cambio coherente)
MAX_DIST_CENTRO_KM = 1.0
MAX_DIF_INCIDENCIA = 1.0   # grados
MAX_DIF_SQUINT = 1.0       # grados

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED), region_name="us-west-2")


# ---------------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------------
def listar_todo(prefijo):
    """Devuelve todas las claves (key, tamaño, fecha) bajo un prefijo."""
    claves = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
        for o in page.get("Contents", []):
            claves.append((o["Key"], o["Size"], o["LastModified"]))
    return claves


def aplanar(d, padre=""):
    """Aplana un JSON anidado: {'a': {'b': 1}} -> {'a.b': 1}. Las listas usan índice."""
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(aplanar(v, f"{padre}.{k}" if padre else k))
    elif isinstance(d, list):
        for i, v in enumerate(d):
            out.update(aplanar(v, f"{padre}[{i}]"))
    else:
        out[padre] = d
    return out


def buscar(plano, *patrones):
    """Primer valor cuya clave (en minúsculas) contenga todos los fragmentos de algún patrón.
    Robusto ante cambios de nombre entre versiones del esquema de metadatos de Umbra."""
    for patron in patrones:
        frag = [p.lower() for p in patron.split("&")]
        for k, v in plano.items():
            kl = k.lower()
            if all(f in kl for f in frag):
                return v
    return None


def tipo_producto(key):
    k = key.lower()
    if k.endswith((".tif", ".tiff")) and "gec" in k:
        return "GEC"
    if "sicd" in k:
        return "SICD"
    if "sidd" in k:
        return "SIDD"
    if k.endswith(".cphd") or "cphd" in k:
        return "CPHD"
    if k.endswith(".json"):
        return "JSON"
    return "OTRO"


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


# ---------------------------------------------------------------------------
# 1. Listado completo
# ---------------------------------------------------------------------------
print("Listando archivos del bucket...")
todas = listar_todo(PREFIJO)
print(f"  {len(todas)} objetos encontrados")

# Agrupar archivos por carpeta de adquisición (directorio que contiene el archivo)
por_carpeta = defaultdict(list)
for key, size, fecha in todas:
    por_carpeta[os.path.dirname(key)].append((key, size, fecha))

pd.DataFrame(todas, columns=["key", "bytes", "last_modified"]).to_csv(
    f"{SALIDA}/umbra_listado_completo.csv", index=False)

# ---------------------------------------------------------------------------
# 2-3. Leer metadatos de cada adquisición
# ---------------------------------------------------------------------------
filas, huellas = [], []
for carpeta, archivos in sorted(por_carpeta.items()):
    jsons = [a for a in archivos if a[0].lower().endswith("metadata.json")] or \
            [a for a in archivos if a[0].lower().endswith(".stac.v2.json")] or \
            [a for a in archivos if a[0].lower().endswith(".json")]
    if not jsons:
        continue
    key_json = jsons[0][0]
    try:
        meta = json.loads(s3.get_object(Bucket=BUCKET, Key=key_json)["Body"].read())
    except Exception as e:
        print(f"  [!] No se pudo leer {key_json}: {e}")
        continue
    p = aplanar(meta)

    productos = sorted({tipo_producto(a[0]) for a in archivos} - {"JSON", "OTRO"})
    gb = sum(a[1] for a in archivos) / 1e9
    partes = carpeta.replace(PREFIJO, "").split("/")

    lat = buscar(p, "scenecenter&lat", "center&lat", "lla[0]")
    lon = buscar(p, "scenecenter&lon", "center&lon", "lla[1]")

    # --- Formato STAC (los archivos *.stac.v2.json de esta carpeta) ---
    es_stac = isinstance(meta, dict) and "properties" in meta and "geometry" in meta
    if es_stac:
        from shapely.geometry import shape
        pr = meta["properties"]
        try:
            c = shape(meta["geometry"]).centroid
            lon, lat = c.x, c.y
        except Exception:
            pass
        huellas.append({"type": "Feature", "geometry": meta["geometry"],
                        "properties": {"task_id": partes[0],
                                       "inicio_utc": pr.get("datetime") or pr.get("start_datetime")}})
        stac_vals = {
            "inicio_utc": pr.get("start_datetime") or pr.get("datetime"),
            "fin_utc": pr.get("end_datetime"),
            "satelite": pr.get("platform") or pr.get("constellation"),
            "modo": pr.get("sar:instrument_mode"),
            "incidencia_deg": pr.get("view:incidence_angle"),
            "azimut_deg": pr.get("view:azimuth"),
            "direccion_mirada": pr.get("sar:observation_direction"),
            "orbita": pr.get("sat:orbit_state"),
            "res_rango_m": pr.get("sar:resolution_range"),
            "res_azimut_m": pr.get("sar:resolution_azimuth"),
            "polarizacion": str(pr.get("sar:polarizations")),
            "squint_deg": buscar(aplanar(pr), "squint"),
            "graze_deg": buscar(aplanar(pr), "graz"),
        }
    fila = {
        "task_id": partes[0],
        "carpeta": "/".join(partes[1:]),
        "inicio_utc": buscar(p, "startatutc", "start&utc", "starttime"),
        "fin_utc": buscar(p, "endatutc", "end&utc", "endtime"),
        "satelite": buscar(p, "satellitename", "satellite"),
        "modo": buscar(p, "imagingmode", "mode"),
        "incidencia_deg": buscar(p, "angleincidence", "incidence"),
        "squint_deg": buscar(p, "anglesquint", "squint"),
        "graze_deg": buscar(p, "graze"),
        "azimut_deg": buscar(p, "angleazimuth", "azimuth&angle"),
        "direccion_mirada": buscar(p, "observationdirection", "lookdirection", "sideoftrack"),
        "orbita": buscar(p, "orbitdirection", "passdirection"),
        "res_rango_m": buscar(p, "resolutionrange", "range&resolution"),
        "res_azimut_m": buscar(p, "resolutionazimuth", "azimuth&resolution"),
        "polarizacion": str(buscar(p, "polarization")),
        "lat_centro": lat,
        "lon_centro": lon,
        "productos": ",".join(productos),
        "tamano_GB": round(gb, 2),
        "metadata_key": key_json,
    }
    if es_stac:
        fila.update({k: v for k, v in stac_vals.items() if v is not None})
        fila["lat_centro"], fila["lon_centro"] = lat, lon
    filas.append(fila)
    if es_stac:
        continue

    # Huella (polígono) si viene en los metadatos como lista de [lon, lat] o GeoJSON
    for k in ("footprintPolygonLla", "footprint", "geometry"):
        if k in meta or any(k.lower() in kk.lower() for kk in p):
            try:
                geo = meta.get(k) or meta["collects"][0].get(k)
                if geo:
                    huellas.append({"type": "Feature", "geometry": geo,
                                    "properties": {"task_id": fila["task_id"], "inicio_utc": fila["inicio_utc"]}})
            except Exception:
                pass
            break

df = pd.DataFrame(filas)
for c in ["incidencia_deg", "squint_deg", "graze_deg", "azimut_deg", "res_rango_m",
          "res_azimut_m", "lat_centro", "lon_centro"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["inicio_utc"] = pd.to_datetime(df["inicio_utc"], errors="coerce", utc=True)
df = df.sort_values("inicio_utc").reset_index(drop=True)

df.to_csv(f"{SALIDA}/umbra_inventario.csv", index=False)
print(f"\n{len(df)} adquisiciones con metadatos -> {SALIDA}/umbra_inventario.csv")
print(df[["task_id", "inicio_utc", "satelite", "incidencia_deg", "squint_deg",
          "direccion_mirada", "lat_centro", "lon_centro", "productos"]].to_string())

if huellas:
    with open(f"{SALIDA}/umbra_huellas.geojson", "w") as f:
        json.dump({"type": "FeatureCollection", "features": huellas}, f)

sismo = pd.Timestamp("2026-06-24T22:04:33Z")
n_pre = (df["inicio_utc"] < sismo).sum()
print(f"\nAdquisiciones ANTES del sismo: {n_pre} | DESPUÉS: {len(df) - n_pre}")
if n_pre == 0:
    print("  -> Todo es post-evento: el 'antes' debe venir de otro sensor "
          "(Planet Q1-2026, Sentinel-1/NISAR) o de comparaciones post-post (remoción de escombros).")

# ---------------------------------------------------------------------------
# 4. Pares con geometría repetida
# ---------------------------------------------------------------------------
pares = []
ok = df.dropna(subset=["lat_centro", "lon_centro", "incidencia_deg"])
for (i, a), (j, b) in combinations(ok.iterrows(), 2):
    d = haversine_km(a.lat_centro, a.lon_centro, b.lat_centro, b.lon_centro)
    dinc = abs(a.incidencia_deg - b.incidencia_deg)
    dsq = abs(np.nan_to_num(a.squint_deg) - np.nan_to_num(b.squint_deg))  # sin squint -> se ignora
    misma_mirada = (a.direccion_mirada == b.direccion_mirada) or pd.isna(a.direccion_mirada)
    # azimut de vista parecido (misma "pasada"); si no hay dato, no se filtra
    if not (pd.isna(a.azimut_deg) or pd.isna(b.azimut_deg)):
        daz = abs((a.azimut_deg - b.azimut_deg + 180) % 360 - 180)
        if daz > 5:
            continue
    if d <= MAX_DIST_CENTRO_KM and dinc <= MAX_DIF_INCIDENCIA and dsq <= MAX_DIF_SQUINT and misma_mirada:
        pares.append({
            "task_a": a.task_id, "fecha_a": a.inicio_utc,
            "task_b": b.task_id, "fecha_b": b.inicio_utc,
            "dias": abs((b.inicio_utc - a.inicio_utc).days),
            "dist_km": round(d, 3), "d_incid": round(dinc, 2), "d_squint": round(dsq, 2),
            "ambos_SICD": "SICD" in a.productos and "SICD" in b.productos,
        })
pares = pd.DataFrame(pares)
pares.to_csv(f"{SALIDA}/umbra_pares_candidatos.csv", index=False)
print(f"\nPares con geometría repetida (candidatos a coherencia/CCD): {len(pares)}")
if len(pares):
    print(pares.to_string())

# ---------------------------------------------------------------------------
# 5. Mapa de adquisiciones
# ---------------------------------------------------------------------------
try:
    import contextily as cx
    import geopandas as gpd
    import matplotlib.pyplot as plt

    g = gpd.GeoDataFrame(ok, geometry=gpd.points_from_xy(ok.lon_centro, ok.lat_centro),
                         crs="EPSG:4326").to_crs("EPSG:3857")
    fig, ax = plt.subplots(figsize=(12, 7))
    dias = (ok["inicio_utc"] - sismo).dt.total_seconds() / 86400
    sc = ax.scatter(g.geometry.x, g.geometry.y, c=dias, cmap="viridis", s=90,
                    edgecolor="k", linewidth=0.6, zorder=3)
    for _, r in g.iterrows():
        ax.annotate(r.task_id[:8], (r.geometry.x, r.geometry.y), xytext=(4, 4),
                    textcoords="offset points", fontsize=7, zorder=4)
    cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, attribution_size=6)
    fig.colorbar(sc, ax=ax, label="Días desde el sismo")
    ax.set_title("Adquisiciones Umbra (banda X) – Sismo de Venezuela 2026\n"
                 "Contains Umbra open data, licensed under CC BY 4.0", fontsize=11)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(f"{SALIDA}/umbra_mapa_adquisiciones.png", dpi=300)
    print(f"\nMapa guardado en {SALIDA}/umbra_mapa_adquisiciones.png")
except Exception as e:
    print(f"(Mapa omitido: {e})")


## Celda 5 · Filtro NISAR (banda L): ¿qué GUNW son cosísmicos? / *Cell 5 · NISAR filter (L-band): which GUNW are coseismic?*

In [ ]:
import json
import os
import re

import asf_search as asf
import geopandas as gpd
import pandas as pd
from shapely.geometry import box, shape

SALIDA = "./nisar_filtrado"
os.makedirs(SALIDA, exist_ok=True)

SISMO = pd.Timestamp("2026-06-24T22:04:33Z")            # Mw 7.2 (USGS)
# Zona completa de ruptura (Yumare -> Caracas) y zona de detalle (La Guaira)
ROI_RUPTURA = box(-68.6, 10.0, -66.4, 11.0)
ROI_GUAIRA = box(-67.3, 10.4, -66.7, 10.8)               # mismo polígono del notebook
FECHA_INI, FECHA_FIN = "2026-05-15", "2026-08-15"

# Tracks publicados por Scripps/UCSD (para comparar con nuestro resultado)
TRACKS_UCSD = {"162A", "061A", "054D", "126D"}

# Footprints de Umbra (generados por umbra_inventario.py), opcional
UMBRA_GEOJSON = "./umbra_inventario/umbra_huellas.geojson"


def fechas_del_nombre(nombre):
    """Extrae todas las fechas YYYYMMDDTHHMMSS del nombre del producto."""
    f = re.findall(r"(\d{8}T\d{6})", nombre)
    return sorted({pd.Timestamp(x, tz="UTC") for x in f})


def clasificar(ref, sec):
    if ref is None or sec is None:
        return "SIN_FECHAS"
    if ref < SISMO < sec:
        return "COSISMICO"
    return "PRE-SISMICO" if sec < SISMO else "POST-SISMICO"


def tamano_mb(b):
    """En Sentinel-1 'bytes' es un número; en NISAR es un dict {archivo: {'bytes': n, ...}}."""
    if b is None:
        return None
    if isinstance(b, (int, float)):
        return round(b / 1e6, 1)
    if isinstance(b, dict):
        total = 0
        for v in b.values():
            if isinstance(v, dict):
                total += v.get("bytes", 0) or 0
            elif isinstance(v, (int, float)):
                total += v
        return round(total / 1e6, 1) if total else None
    return None


def buscar_nisar(nivel):
    try:
        return asf.geo_search(platform=[asf.PLATFORM.NISAR], intersectsWith=ROI_RUPTURA.wkt,
                              start=FECHA_INI, end=FECHA_FIN, processingLevel=nivel)
    except Exception as e:
        print(f"[!] Búsqueda {nivel} falló: {e}")
        return []


# ---------------------------------------------------------------------------
# 1. Búsqueda
# ---------------------------------------------------------------------------
umbra = None
if os.path.exists(UMBRA_GEOJSON):
    umbra = gpd.read_file(UMBRA_GEOJSON)
    print(f"Umbra: {len(umbra)} huellas cargadas para cruzar.")

filas = []
for nivel in ["GUNW", "RSLC"]:
    res = buscar_nisar(nivel)
    print(f"NISAR {nivel}: {len(res)} productos en la zona de ruptura")
    for r in res:
        p = r.properties
        nombre = p.get("sceneName") or p.get("fileID") or p.get("fileName", "")
        geom = shape(r.geometry)
        fechas = fechas_del_nombre(nombre)

        if nivel == "GUNW":
            # El GUNW lleva fecha de referencia y secundaria en el nombre (inicio/fin de cada una)
            ref = fechas[0] if fechas else None
            sec = fechas[-1] if len(fechas) > 1 else None
            if ref is not None and sec is not None and (sec - ref).days < 1:
                sec = None                       # solo encontró una adquisición -> revisar
        else:
            ref = pd.Timestamp(p["startTime"]) if p.get("startTime") else None
            sec = None

        direccion = (p.get("flightDirection") or "").upper()[:1]  # A / D
        track = f"{int(p['pathNumber']):03d}{direccion}" if p.get("pathNumber") is not None else None
        inter_guaira = geom.intersection(ROI_GUAIRA).area / ROI_GUAIRA.area * 100

        toca_umbra = None
        if umbra is not None:
            toca_umbra = int(umbra.intersects(geom).sum())

        filas.append({
            "nivel": nivel,
            "nombre": nombre,
            "track": track,
            "frame": p.get("frameNumber"),
            "direccion": p.get("flightDirection"),
            "fecha_ref": ref,
            "fecha_sec": sec,
            "dias": (sec - ref).days if (ref is not None and sec is not None) else None,
            "tipo": clasificar(ref, sec) if nivel == "GUNW" else
                    ("PRE" if ref is not None and ref < SISMO else "POST"),
            "cubre_LaGuaira_%": round(inter_guaira, 1),
            "escenas_Umbra_que_toca": toca_umbra,
            "en_UCSD": track in TRACKS_UCSD if track else None,
            "MB": tamano_mb(p.get("bytes")),
            "url": p.get("url"),
            "geometry": geom,
        })

if not filas:
    raise SystemExit("No se encontraron productos NISAR. Revise autenticación / fechas.")

gdf = gpd.GeoDataFrame(filas, geometry="geometry", crs="EPSG:4326")
gdf = gdf.sort_values(["nivel", "track", "fecha_ref"]).reset_index(drop=True)
gdf.drop(columns="geometry").to_csv(f"{SALIDA}/nisar_productos.csv", index=False)
gdf[["nivel", "nombre", "track", "tipo", "geometry"]].to_file(
    f"{SALIDA}/nisar_huellas.geojson", driver="GeoJSON")

# ---------------------------------------------------------------------------
# 2. Resumen y filtro
# ---------------------------------------------------------------------------
gunw = gdf[gdf.nivel == "GUNW"]
print("\n=== GUNW por tipo ===")
print(gunw.groupby(["tipo", "track"]).size().to_string())

cos = gunw[(gunw.tipo == "COSISMICO") & (gunw["cubre_LaGuaira_%"] > 0)].copy()
cos = cos.sort_values(["dias", "cubre_LaGuaira_%"], ascending=[True, False])
cos.drop(columns="geometry").to_csv(f"{SALIDA}/nisar_cosismicos.csv", index=False)

print(f"\n=== {len(cos)} GUNW COSÍSMICOS que tocan La Guaira ===")
cols = ["track", "frame", "fecha_ref", "fecha_sec", "dias", "cubre_LaGuaira_%",
        "escenas_Umbra_que_toca", "en_UCSD", "nombre"]
print(cos[cols].to_string(index=False))

# Mejor par por track: menor línea base temporal y mayor cobertura
mejor = cos.groupby("track", as_index=False).first()
with open(f"{SALIDA}/nisar_lista_descarga.txt", "w") as f:
    for u in mejor.url.dropna():
        f.write(u + "\n")
print(f"\nMejor GUNW por track ({len(mejor)}) -> {SALIDA}/nisar_lista_descarga.txt")
print(mejor[["track", "fecha_ref", "fecha_sec", "dias", "cubre_LaGuaira_%"]].to_string(index=False))

asc = mejor.direccion.str.upper().str.startswith("A").any()
desc = mejor.direccion.str.upper().str.startswith("D").any()
print(f"\n¿Geometría ascendente y descendente? ASC={asc}  DESC={desc}"
      + ("  -> se puede descomponer en Este/Vertical" if asc and desc else
         "  -> falta una geometría para la descomposición 2D"))
faltan = TRACKS_UCSD - set(mejor.track.dropna())
if faltan:
    print(f"Tracks usados por UCSD que NO aparecen como cosísmicos aquí: {sorted(faltan)}"
          " (puede que solo existan como RSLC: habría que formar el par uno mismo)")

sin_f = gunw[gunw.tipo == "SIN_FECHAS"]
if len(sin_f):
    print(f"\n[!] {len(sin_f)} GUNW sin dos fechas legibles en el nombre: revisarlos a mano.")

# ---------------------------------------------------------------------------
# 3. Mapa de cobertura
# ---------------------------------------------------------------------------
try:
    import contextily as cx
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(13, 7))
    g3857 = gdf.to_crs("EPSG:3857")
    colores = {"COSISMICO": "#d62728", "PRE-SISMICO": "#7f7f7f", "POST-SISMICO": "#1f77b4"}
    for tipo, color in colores.items():
        sub = g3857[(g3857.nivel == "GUNW") & (g3857.tipo == tipo)]
        if len(sub):
            sub.boundary.plot(ax=ax, color=color, linewidth=1.2, label=f"GUNW {tipo.lower()}")
    gpd.GeoSeries([ROI_GUAIRA], crs="EPSG:4326").to_crs("EPSG:3857").boundary.plot(
        ax=ax, color="yellow", linewidth=2.5, label="ROI La Guaira")
    if umbra is not None:
        umbra.to_crs("EPSG:3857").plot(ax=ax, color="cyan", alpha=0.6, label="Umbra (X)")
    cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, attribution_size=6)
    ax.legend(loc="lower left", fontsize=9)
    ax.set_title("Cobertura NISAR (banda L) y Umbra (banda X) – Sismo de Venezuela 24/06/2026")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(f"{SALIDA}/nisar_mapa_cobertura.png", dpi=300)
    plt.show()
except Exception as e:
    print(f"(Mapa omitido: {e})")

# ---------------------------------------------------------------------------
# 4. Descarga SOLO de los cosísmicos seleccionados (descomentar)
# ---------------------------------------------------------------------------
# urls = open(f"{SALIDA}/nisar_lista_descarga.txt").read().split()
# asf.download_urls(urls=urls, path="./datos_nisar_vargas", session=session, processes=4)


## Celda 6 · Inventario multisensor + línea de tiempo / *Cell 6 · Multi-sensor inventory + timeline figure*

In [ ]:
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

SALIDA = "./inventario"
os.makedirs(SALIDA, exist_ok=True)

SISMO = pd.Timestamp("2026-06-24T22:04:33Z")
ROI_WKT = "POLYGON((-68.6 10.0, -66.4 10.0, -66.4 11.0, -68.6 11.0, -68.6 10.0))"  # zona de ruptura
INI, FIN = "2026-06-01", "2026-07-31"

# Par Sentinel-1 ya procesado por usted en HyP3 (del notebook)
S1_PAR_PROCESADO = ("S1A_IW_SLC__1SDV_20260618T224250_20260618T224320_065030_083236_48BF",
                    "S1D_IW_SLC__1SDV_20260625T224209_20260625T224239_003399_005FB5_8B0A")

PLANET_PARQUET = ("https://data.source.coop/planet/disasterdata/"
                  "venezuela-earthquake-2026-06-24/{col}/items.parquet")

COLUMNAS = ["sensor", "banda", "lambda_cm", "tipo_producto", "id", "fecha", "fecha_sec",
            "relacion_sismo", "track_o_sitio", "direccion", "resolucion_m", "licencia",
            "fuente", "usado_en_notebook"]
filas = []


def rel(fecha, fecha_sec=None):
    """PRE / POST / COSISMICO respecto al sismo."""
    if fecha_sec is not None and pd.notna(fecha_sec):
        if fecha < SISMO < fecha_sec:
            return "COSISMICO"
        return "PRE" if fecha_sec < SISMO else "POST"
    return "PRE" if fecha < SISMO else "POST"


# ---------------------------------------------------------------------------
# 1. Sentinel-1 (banda C)
# ---------------------------------------------------------------------------
try:
    import asf_search as asf
    res = asf.geo_search(platform=[asf.PLATFORM.SENTINEL1], intersectsWith=ROI_WKT,
                         start=INI, end=FIN, processingLevel=asf.PRODUCT_TYPE.SLC)
    for r in res:
        p = r.properties
        f = pd.Timestamp(p["startTime"])
        filas.append(dict(
            sensor=p.get("platform", "Sentinel-1"), banda="C", lambda_cm=5.55,
            tipo_producto="SLC", id=p["sceneName"], fecha=f, fecha_sec=pd.NaT,
            relacion_sismo=rel(f), track_o_sitio=f"T{int(p['pathNumber']):03d}",
            direccion=p.get("flightDirection"), resolucion_m=20,
            licencia="Copernicus (libre)", fuente="ASF",
            usado_en_notebook=p["sceneName"] in S1_PAR_PROCESADO))
    print(f"Sentinel-1: {len(res)} escenas SLC")
except Exception as e:
    print(f"[!] Sentinel-1 omitido: {e}")

# Par interferométrico procesado (HyP3, 80 m)
ref = pd.Timestamp("2026-06-18T22:42:50Z")
sec = pd.Timestamp("2026-06-25T22:42:09Z")
filas.append(dict(sensor="Sentinel-1 (par HyP3)", banda="C", lambda_cm=5.55,
                  tipo_producto="INSAR_GAMMA (unw, corr, dem, inc)",
                  id="Sismo_LaGuaira_S1_Path33", fecha=ref, fecha_sec=sec,
                  relacion_sismo=rel(ref, sec), track_o_sitio="T033", direccion="ASCENDING",
                  resolucion_m=80, licencia="Copernicus (libre)", fuente="ASF HyP3",
                  usado_en_notebook=True))

# ---------------------------------------------------------------------------
# 2. NISAR banda L (GUNW)
# ---------------------------------------------------------------------------
csv_nisar = "./nisar_filtrado/nisar_productos.csv"
if os.path.exists(csv_nisar):
    nz = pd.read_csv(csv_nisar, parse_dates=["fecha_ref", "fecha_sec"])
    nz = nz[nz.nivel == "GUNW"]
    descargados = set(os.listdir("./datos_nisar_vargas")) if os.path.isdir("./datos_nisar_vargas") else set()
    for _, r in nz.iterrows():
        filas.append(dict(
            sensor="NISAR-L", banda="L", lambda_cm=23.84, tipo_producto="GUNW (HH)",
            id=r.nombre, fecha=r.fecha_ref, fecha_sec=r.fecha_sec,
            relacion_sismo=rel(r.fecha_ref, r.fecha_sec), track_o_sitio=f"T{r.track}",
            direccion=r.direccion, resolucion_m=80, licencia="NASA (libre)", fuente="ASF",
            usado_en_notebook=any(str(r.nombre) in d for d in descargados)))
    print(f"NISAR-L: {len(nz)} GUNW (desde {csv_nisar})")
else:
    print("[!] NISAR-L: ejecute primero nisar_filtrar.py para clasificar los GUNW.")

# ---------------------------------------------------------------------------
# 3. NISAR banda S (manual, Bhoonidhi)
# ---------------------------------------------------------------------------
# Si encuentra escenas en https://bhoonidhi.nrsc.gov.in, agréguelas aquí:
NISAR_S = [
    # ("NISAR_S2_PR_GCOV_...", "2026-07-xxTxx:xx:xxZ", "TxxxA"),
]
for nombre, fecha, track in NISAR_S:
    f = pd.Timestamp(fecha)
    filas.append(dict(sensor="NISAR-S", banda="S", lambda_cm=9.4, tipo_producto="GCOV/GSLC",
                      id=nombre, fecha=f, fecha_sec=pd.NaT, relacion_sismo=rel(f),
                      track_o_sitio=track, direccion=None, resolucion_m=None,
                      licencia="ISRO (registro)", fuente="Bhoonidhi", usado_en_notebook=False))

# ---------------------------------------------------------------------------
# 4. Umbra banda X
# ---------------------------------------------------------------------------
csv_umbra = "./umbra_inventario/umbra_inventario.csv"
if os.path.exists(csv_umbra):
    um = pd.read_csv(csv_umbra, parse_dates=["inicio_utc"])
    for _, r in um.iterrows():
        filas.append(dict(
            sensor=f"Umbra ({r.satelite})" if pd.notna(r.satelite) else "Umbra",
            banda="X", lambda_cm=3.1, tipo_producto=r.productos, id=r.task_id,
            fecha=r.inicio_utc, fecha_sec=pd.NaT, relacion_sismo=rel(r.inicio_utc),
            track_o_sitio=f"{r.lat_centro:.3f},{r.lon_centro:.3f}" if pd.notna(r.lat_centro) else None,
            direccion=r.direccion_mirada, resolucion_m=r.res_rango_m,
            licencia="CC BY 4.0", fuente="Umbra Open Data (AWS)", usado_en_notebook=False))
    print(f"Umbra: {len(um)} adquisiciones")
else:
    print("[!] Umbra: ejecute primero umbra_inventario.py")

# ---------------------------------------------------------------------------
# 5. Planet (óptico) – STAC-GeoParquet en source.coop
# ---------------------------------------------------------------------------
try:
    import duckdb
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    for col in ["post-event", "pre-event"]:
        df = con.execute(f"SELECT * FROM read_parquet('{PLANET_PARQUET.format(col=col)}')").df()
        # El índice puede traer las propiedades como columnas o dentro de 'properties'
        if "properties" in df.columns:
            df = pd.concat([df.drop(columns="properties"),
                            pd.json_normalize(df["properties"])], axis=1)
        for _, r in df.iterrows():
            f = pd.Timestamp(r.get("datetime") or r.get("start_datetime"))
            if f.tzinfo is None:
                f = f.tz_localize("UTC")
            filas.append(dict(
                sensor=f"Planet {r.get('constellation') or ('PlanetScope mosaico' if col == 'pre-event' else '')}".strip(),
                banda="Óptico", lambda_cm=None,
                tipo_producto="visual/analytic/udm2" if col == "post-event" else "mosaico trimestral RGB",
                id=r.get("id"), fecha=f, fecha_sec=pd.NaT,
                relacion_sismo="PRE" if col == "pre-event" else rel(f),
                track_o_sitio=r.get("location") or r.get("location_slug") or r.get("collection"),
                direccion=None, resolucion_m=r.get("gsd"),
                licencia="CC BY-NC 4.0 (no comercial)", fuente="Planet Crisis Response / source.coop",
                usado_en_notebook=False))
        print(f"Planet {col}: {len(df)} ítems")
except Exception as e:
    print(f"[!] Planet omitido: {e}")

# ---------------------------------------------------------------------------
# 6. Tabla maestra y resumen
# ---------------------------------------------------------------------------
inv = pd.DataFrame(filas, columns=COLUMNAS)
for c in ["fecha", "fecha_sec"]:
    inv[c] = pd.to_datetime(inv[c], utc=True, errors="coerce")
inv["dias_desde_sismo"] = ((inv.fecha - SISMO).dt.total_seconds() / 86400).round(2)
inv = inv.sort_values(["banda", "sensor", "fecha"]).reset_index(drop=True)
inv.to_csv(f"{SALIDA}/inventario_multisensor.csv", index=False)

resumen = (inv.assign(familia=inv.sensor.str.replace(r"^Sentinel-1\w?", "Sentinel-1", regex=True).str.split(" ").str[0])
           .pivot_table(index=["familia", "banda"], columns="relacion_sismo",
                        values="id", aggfunc="count", fill_value=0))
resumen.to_csv(f"{SALIDA}/resumen_sensores.csv")
print("\n=== RESUMEN ===")
print(resumen.to_string())

# ---------------------------------------------------------------------------
# 7. Figura: línea de tiempo de adquisiciones por sensor (para el artículo)
# ---------------------------------------------------------------------------
orden = ["Umbra", "Sentinel-1", "NISAR-S", "NISAR-L", "ALOS-2", "Planet"]
color = {"X": "#4C72B0", "C": "#DD8452", "S": "#8172B3", "L": "#55A868", "Óptico": "#937860"}
inv["familia"] = inv.sensor.str.split(" ").str[0].replace({"Sentinel-1A": "Sentinel-1",
                                                           "Sentinel-1C": "Sentinel-1",
                                                           "Sentinel-1D": "Sentinel-1"})
familias = [f for f in orden if f in set(inv.familia)]
ypos = {f: i for i, f in enumerate(familias)}

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
fig, ax = plt.subplots(figsize=(10, 0.7 * len(familias) + 1.6))
ventana = inv[(inv.fecha > SISMO - pd.Timedelta(days=30)) & (inv.fecha < SISMO + pd.Timedelta(days=30))]
for _, r in ventana.iterrows():
    y = ypos[r.familia]
    c = color.get(r.banda, "k")
    if pd.notna(r.fecha_sec):   # par interferométrico -> segmento
        ax.plot([r.fecha, r.fecha_sec], [y + 0.18, y + 0.18], color=c, lw=2.2, alpha=0.8,
                solid_capstyle="round")
    else:
        ax.plot(r.fecha, y, "o", ms=6, mfc=c, mec="white", mew=0.6, alpha=0.9)
ax.axvline(SISMO, color="#C44E52", lw=1.5, ls="--")
ax.text(SISMO, len(familias) - 0.4, "  Mw 7.2 + 7.5\n  24 jun 2026", color="#C44E52",
        va="top", fontsize=9)
ax.set_yticks(range(len(familias)))
ax.set_yticklabels([f"{f}  ({ventana[ventana.familia == f].banda.iloc[0] if (ventana.familia == f).any() else ''})"
                    for f in familias])
ax.set_ylim(-0.6, len(familias) - 0.3)
ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.grid(axis="x", color="0.9")
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.set_title("Adquisiciones disponibles por sensor (±30 días del sismo)\n"
             "Puntos: escenas · Segmentos: pares interferométricos", fontsize=11, loc="left")
plt.tight_layout()
plt.savefig(f"{SALIDA}/fig_linea_tiempo_sensores.png", dpi=300)
plt.savefig(f"{SALIDA}/fig_linea_tiempo_sensores.pdf")
plt.show()
print(f"\nTabla y figura guardadas en {SALIDA}/")


## Celda 7 · Umbra × Planet en los sitios de daño / *Cell 7 · Umbra × Planet at damage sites*
🇪🇸 Versión de bajo consumo de memoria: procesa **un sitio a la vez** (Catia La Mar, Playa Grande, La Guaira, Caraballeda, Tanaguarena, Naiguatá). Paneles: Planet antes, Planet después, Umbra amplitud (filtro Lee), Umbra textura (CV local) y, si hay dos miradas, composición multi-aspecto. Si falta RAM: baje `LADO_M` a 600 o use `SITIOS_A_PROCESAR = ["Catia_La_Mar"]`.  
🇬🇧 Low-memory version: processes **one site at a time**. Panels: Planet before, Planet after, Umbra amplitude (Lee filter), Umbra texture (local CV) and, when two looks exist, a multi-aspect composite. If RAM runs out: set `LADO_M = 600` or `SITIOS_A_PROCESAR = ["Catia_La_Mar"]`.

In [ ]:
import gc
import json
import os

import geopandas as gpd
import matplotlib
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from scipy.ndimage import uniform_filter
from shapely.geometry import Point

# ---------------------------------------------------------------------------
# Configuración / Configuration
# ---------------------------------------------------------------------------
SALIDA = "./umbra_planet_sitios"
os.makedirs(SALIDA, exist_ok=True)
CRS = "EPSG:32619"
LADO_M = 1000          # lado del recorte (m) / chip side (m)   -> bajar a 600 si falta RAM
RES_M = 0.5            # resolución común / common resolution (m)
HORA_LOCAL = "America/Caracas"
UMBRA_DIR = "./umbra_inventario"
UMBRA_HTTPS = "https://umbra-open-data-catalog.s3.us-west-2.amazonaws.com/"
PLANET_BASE = "https://data.source.coop/planet/disasterdata/venezuela-earthquake-2026-06-24"
PLANET_PRE = f"{PLANET_BASE}/pre-event/quarterly-mosaic/quarterly-mosaic-2026Q1.tif"
PLANET_POST_PARQUET = f"{PLANET_BASE}/post-event/items.parquet"

# Sitios de daño (coordenadas aproximadas del centro urbano; ajustar a edificios concretos)
# Damage sites (approximate town-centre coordinates; refine to specific buildings)
SITIOS = {
    "Catia_La_Mar":          (10.6055, -67.0324),
    "Playa_Grande_Urimare":  (10.6100, -67.0103),
    "La_Guaira":             (10.6000, -66.9331),
    "Caraballeda":           (10.6122, -66.8519),
    "Tanaguarena":           (10.6150, -66.8200),
    "Naiguata":              (10.6200, -66.7400),
}
EPICENTRO_MW75 = (10.622, -67.194)   # USGS, 17 km W de Catia La Mar
SITIOS_A_PROCESAR = None             # None = todos; o p. ej. ["Catia_La_Mar"] para ir por partes

os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif,.tiff")
matplotlib.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9})


# ---------------------------------------------------------------------------
# Utilidades / Utilities
# ---------------------------------------------------------------------------
def leer(url, cx, cy, bandas, lado=LADO_M, res=RES_M):
    n = int(lado / res)
    tr = from_origin(cx - lado / 2, cy + lado / 2, res, res)
    with rasterio.open(url) as src, WarpedVRT(src, crs=CRS, transform=tr, width=n, height=n,
                                              resampling=Resampling.bilinear) as vrt:
        b = [i for i in bandas if i <= vrt.count]
        return np.ma.filled(vrt.read(b, masked=True).astype("float32"), np.nan)


def rgb(arr):
    out = np.ones(arr.shape[1:] + (3,), dtype="float32")
    for i in range(min(3, arr.shape[0])):
        b = arr[i]
        v = b[np.isfinite(b) & (b > 0)]
        if v.size:
            lo, hi = np.percentile(v, [2, 98])
            out[..., i] = np.clip((b - lo) / max(hi - lo, 1e-6), 0, 1)
    return out


def lee(amp, w=5):
    """Filtro de Lee sobre intensidad / Lee filter on intensity."""
    I = np.nan_to_num(amp, nan=0.0) ** 2
    m = uniform_filter(I, w)
    m2 = uniform_filter(I * I, w)
    var = np.maximum(m2 - m * m, 0)
    ruido = np.nanmean(var / np.maximum(m * m, 1e-12))
    k = var / np.maximum(var + m * m * ruido, 1e-12)
    out = m + k * (I - m)
    out[~np.isfinite(amp)] = np.nan
    return np.sqrt(np.maximum(out, 0))


def db(amp):
    a = amp.copy()
    a[a <= 0] = np.nan
    return 20 * np.log10(a)


def cv_local(amp, w=9):
    """Coeficiente de variación local / local coefficient of variation."""
    a = np.nan_to_num(amp, nan=0.0)
    m = uniform_filter(a, w)
    s = np.sqrt(np.maximum(uniform_filter(a * a, w) - m * m, 0))
    cv = s / np.maximum(m, 1e-6)
    cv[~np.isfinite(amp)] = np.nan
    return cv


def estirar(x, p=(2, 98)):
    v = x[np.isfinite(x)]
    if v.size == 0:
        return np.zeros_like(x)
    lo, hi = np.percentile(v, p)
    return np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)


def barra(ax, n, metros=200):
    L = metros / RES_M
    x0, y0 = n * 0.05, n * 0.93
    ax.plot([x0, x0 + L], [y0, y0], color="white", lw=4, solid_capstyle="butt")
    ax.plot([x0, x0 + L], [y0, y0], color="black", lw=1.5, solid_capstyle="butt")
    ax.text(x0 + L / 2, y0 - n * 0.03, f"{metros} m", color="white", ha="center", fontsize=7.5,
            weight="bold", path_effects=[pe.withStroke(linewidth=2, foreground="black")])


# ---------------------------------------------------------------------------
# 1. Catálogos / Catalogues
# ---------------------------------------------------------------------------
umbra = gpd.read_file(f"{UMBRA_DIR}/umbra_huellas.geojson").set_crs("EPSG:4326", allow_override=True)
inv = pd.read_csv(f"{UMBRA_DIR}/umbra_inventario.csv", parse_dates=["inicio_utc"])
lst = pd.read_csv(f"{UMBRA_DIR}/umbra_listado_completo.csv")
gec = lst[lst.key.str.contains("GEC", case=False) & lst.key.str.lower().str.endswith((".tif", ".tiff"))]
umbra = umbra[["task_id", "geometry"]].merge(
    inv[["task_id", "inicio_utc", "satelite", "incidencia_deg", "direccion_mirada"]], on="task_id")
umbra["url"] = umbra.task_id.map(lambda t: UMBRA_HTTPS + gec[gec.key.str.contains(t)].key.iloc[0]
                                 if gec.key.str.contains(t).any() else None)

pq = f"{SALIDA}/planet_post_items.parquet"
if not os.path.exists(pq):
    r = requests.get(PLANET_POST_PARQUET, timeout=120); r.raise_for_status()
    open(pq, "wb").write(r.content)
planet = gpd.read_parquet(pq).to_crs("EPSG:4326")


def href(assets):
    if isinstance(assets, str):
        assets = json.loads(assets)
    for k in ("visual", "analytic"):
        a = assets.get(k) if isinstance(assets, dict) else None
        if a:
            return a.get("href") if isinstance(a, dict) else a


def absoluta(fila, rel):
    from urllib.parse import urljoin
    if not rel:
        return None
    if str(rel).startswith("http"):
        return rel
    cand = []
    links = fila.get("links")
    if isinstance(links, str):
        links = json.loads(links)
    if links is not None:
        for l in list(links):
            if isinstance(l, dict) and l.get("rel") == "self" and str(l.get("href", "")).startswith("http"):
                cand.append(urljoin(l["href"], rel))
    arch = str(rel).replace("./", "", 1)
    for lugar in (fila.get("location_slug"), fila.get("location"), fila.get("collection")):
        if isinstance(lugar, str) and lugar:
            cand += [f"{PLANET_BASE}/post-event/{lugar}/items/{fila['id']}/{arch}",
                     f"{PLANET_BASE}/post-event/{lugar}/{fila['id']}/{arch}"]
    cand.append(f"{PLANET_BASE}/post-event/items/{fila['id']}/{arch}")
    for u in dict.fromkeys(cand):
        try:
            if requests.head(u, timeout=20, allow_redirects=True).status_code == 200:
                return u
        except requests.RequestException:
            pass
    return None


planet["url"] = [absoluta(f, href(a)) for f, a in zip(planet.to_dict("records"), planet["assets"])]
planet["fecha"] = pd.to_datetime(planet["datetime"], utc=True)
print(f"Planet: {planet.url.notna().sum()}/{len(planet)} URLs válidas / valid URLs")
if planet.url.notna().any():
    print("  ej.:", planet.url.dropna().iloc[0])

# ---------------------------------------------------------------------------
# 2. Cobertura por sitio / Coverage per site
# ---------------------------------------------------------------------------
filas = []
for nombre, (lat, lon) in SITIOS.items():
    p = Point(lon, lat)
    u = umbra[umbra.contains(p)].sort_values("inicio_utc")
    pl = planet[planet.contains(p) & planet.url.notna()].sort_values("gsd" if "gsd" in planet else "fecha")
    filas.append(dict(sitio=nombre, lat=lat, lon=lon, n_umbra=len(u),
                      miradas=",".join(sorted(set(u.direccion_mirada.dropna()))),
                      n_planet_post=len(pl), umbra_tasks=",".join(t[:8] for t in u.task_id),
                      planet_ids=",".join(str(i)[:22] for i in pl["id"])))
cob = pd.DataFrame(filas)
cob.to_csv(f"{SALIDA}/cobertura_sitios.csv", index=False)
print("\n=== Cobertura por sitio / Coverage per site ===")
print(cob[["sitio", "n_umbra", "miradas", "n_planet_post"]].to_string(index=False))
gpd.GeoDataFrame(cob, geometry=gpd.points_from_xy(cob.lon, cob.lat), crs="EPSG:4326").to_file(
    f"{SALIDA}/sitios_danio.geojson", driver="GeoJSON")

# ---------------------------------------------------------------------------
# 3. Figuras por sitio (una a la vez) / Figures per site (one at a time)
# ---------------------------------------------------------------------------
from pyproj import Transformer
tf = Transformer.from_crs(4326, 32619, always_xy=True)
hechos = []
for nombre, (lat, lon) in SITIOS.items():
    if SITIOS_A_PROCESAR and nombre not in SITIOS_A_PROCESAR:
        continue
    p = Point(lon, lat)
    u = umbra[umbra.contains(p) & umbra.url.notna()].sort_values("inicio_utc")
    pl = planet[planet.contains(p) & planet.url.notna()]
    if "gsd" in pl:
        pl = pl.sort_values("gsd")
    if u.empty:
        print(f"[–] {nombre}: sin Umbra / no Umbra coverage"); continue
    cx, cy = tf.transform(lon, lat)
    try:
        pre = rgb(leer(PLANET_PRE, cx, cy, [1, 2, 3]))
        post = rgb(leer(pl.url.iloc[0], cx, cy, [1, 2, 3])) if len(pl) else None
        a1 = leer(u.url.iloc[0], cx, cy, [1])[0]
        a1f = lee(a1)
        dos_miradas = u.direccion_mirada.nunique() > 1
        if dos_miradas:
            u2 = u[u.direccion_mirada != u.direccion_mirada.iloc[0]].iloc[0]
            a2f = lee(leer(u2.url, cx, cy, [1])[0])
    except Exception as e:
        print(f"[!] {nombre}: {e}"); continue

    paneles = [(pre, None, "(a) Planet antes / before\nQ1-2026 · ~4.8 m")]
    if post is not None:
        t = pl.fecha.iloc[0].tz_convert(HORA_LOCAL)
        paneles.append((post, None, f"(b) Planet después / after\n{t:%d %b %H:%M} local · "
                                    f"{pl['gsd'].iloc[0] if 'gsd' in pl else ''} m"))
    tu = u.inicio_utc.iloc[0].tz_convert(HORA_LOCAL)
    paneles.append((estirar(db(a1f)), "gray", f"(c) Umbra X amplitud / amplitude (dB, Lee)\n"
                                              f"{tu:%d %b %H:%M} local · mirada/look {u.direccion_mirada.iloc[0]}"))
    paneles.append((estirar(cv_local(a1f), (5, 95)), "magma", "(d) Umbra textura / texture\nCV local 9×9"))
    if dos_miradas:
        comp = np.dstack([estirar(db(a1f)), estirar(db(a2f)), estirar(db((a1f + a2f) / 2))])
        comp[~np.isfinite(a1f) | ~np.isfinite(a2f)] = 1
        paneles.append((comp, None, "(e) Umbra multi-aspecto / multi-aspect\nR = izq./left · G = der./right"))

    fig, axs = plt.subplots(1, len(paneles), figsize=(4.2 * len(paneles), 4.6))
    for ax, (img, cmap, tit) in zip(np.atleast_1d(axs), paneles):
        ax.imshow(img, cmap=cmap, interpolation="nearest")
        ax.set_title(tit, loc="left", fontsize=8.5)
        ax.set_xticks([]); ax.set_yticks([])
        barra(ax, img.shape[1])
    fig.suptitle(f"{nombre.replace('_', ' ')}  ({lat:.4f}° N, {lon:.4f}° W) · {LADO_M} m × {LADO_M} m",
                 x=0.01, ha="left", fontsize=10.5, weight="bold")
    fig.text(0.01, 0.01, "Contains Umbra open data, CC BY 4.0 · Imagery © Planet Labs PBC, CC BY-NC 4.0",
             fontsize=6.5, color="0.35")
    plt.tight_layout(rect=(0, 0.03, 1, 0.94))
    ruta = f"{SALIDA}/{nombre}.png"
    plt.savefig(ruta, dpi=200)
    plt.show()
    plt.close(fig)
    hechos.append(ruta)
    print(f"✓ {nombre} -> {ruta}")
    # liberar memoria / free memory
    del pre, post, a1, a1f, paneles
    if dos_miradas:
        del a2f, comp
    gc.collect()

print(f"\nListo / Done: {len(hechos)} sitios. Figuras en {SALIDA}/")


## Celda 8 · (Opcional) Descargar los NISAR cosísmicos de La Guaira / *Cell 8 · (Optional) Download the La Guaira coseismic NISAR*
🇪🇸 Revise primero la tabla de la Celda 5.  
🇬🇧 Check the Cell 5 table first.

In [ ]:
urls = open('./nisar_filtrado/nisar_lista_descarga.txt').read().split()
print(len(urls), 'productos a descargar')
os.makedirs('./datos_nisar_vargas', exist_ok=True)
asf.download_urls(urls=urls, path='./datos_nisar_vargas', session=session, processes=4)
print('Descarga terminada')


## Celda 9 · Pares cosísmicos de Sentinel-1 / *Cell 9 · Sentinel-1 coseismic pairs*

In [ ]:
import asf_search as asf, pandas as pd, hyp3_sdk
from shapely.geometry import box, shape

SISMO = pd.Timestamp("2026-06-24T22:04:33Z")
ROI = box(-67.3, 10.4, -66.7, 10.8)   # La Guaira

res = asf.geo_search(platform=[asf.PLATFORM.SENTINEL1], intersectsWith=ROI.wkt,
                     start="2026-06-01", end="2026-07-15",
                     processingLevel=asf.PRODUCT_TYPE.SLC)
s1 = pd.DataFrame([dict(escena=r.properties["sceneName"],
                        fecha=pd.Timestamp(r.properties["startTime"]),
                        track=int(r.properties["pathNumber"]),
                        dir=r.properties["flightDirection"],
                        geom=shape(r.geometry)) for r in res])
s1["cubre_%"] = s1.geom.map(lambda g: round(g.intersection(ROI).area / ROI.area * 100, 1))

pares = []
for (t, d), g in s1.groupby(["track", "dir"]):
    pre = g[g.fecha < SISMO].sort_values("fecha")
    post = g[g.fecha > SISMO].sort_values("fecha")
    if pre.empty or post.empty:
        continue
    a = pre.iloc[-1]
    post = post[post.geom.map(lambda x: x.intersection(a.geom).area / a.geom.area) > 0.5]
    if post.empty:
        continue
    b = post.iloc[0]
    pares.append(dict(track=f"{t:03d}{d[0]}", ref=a.escena, sec=b.escena,
                      fecha_ref=a.fecha.date(), fecha_sec=b.fecha.date(),
                      dias=(b.fecha - a.fecha).days, cubre_LaGuaira=min(a["cubre_%"], b["cubre_%"])))

pares = pd.DataFrame(pares).sort_values("cubre_LaGuaira", ascending=False)
pares.to_csv("s1_pares_cosismicos.csv", index=False)
print(pares.to_string(index=False))


## Celda 10 · Enviar a HyP3 los pares 033A y 025D (40 m, vectores de mirada) / *Cell 10 · Submit 033A and 025D pairs to HyP3 (40 m, look vectors)*
🇪🇸 Gasta créditos HyP3. Si un trabajo ya existe con ese nombre, no se repite.  
🇬🇧 Uses HyP3 credits. Jobs that already exist with the same name are not resubmitted.

In [ ]:
# Se reprocesan LOS DOS pares (033A y 025D) a ~40 m y CON vectores de mirada.
# El 033A original (80 m) no trae lv_theta/lv_phi, que son necesarios para descomponer Este/Vertical.
hyp3 = hyp3_sdk.HyP3(username=usuario, password=clave)
existentes = {j.name for j in hyp3.find_jobs()}
trabajos = hyp3_sdk.Batch()
for _, p in pares.iterrows():
    nombre = f"S1_{p.track}_cosismico"
    if nombre in existentes:
        print("ya enviado:", nombre)
        continue
    trabajos += hyp3.submit_insar_job(p.ref, p.sec, name=nombre, looks="10x2",
                                      include_inc_map=True, include_dem=True,
                                      include_look_vectors=True)
print(trabajos)
print(hyp3.find_jobs(name="S1_025D_cosismico"), hyp3.find_jobs(name="S1_033A_cosismico"))


## Celda 11 · Descargar todo a Drive / *Cell 11 · Download everything to Drive*
🇪🇸 NISAR (6 cosísmicos), Umbra (26 GEC) y Sentinel-1 HyP3. Se puede volver a correr; la parte de HyP3 espera hasta que los trabajos terminen.  
🇬🇧 NISAR (6 coseismic), Umbra (26 GEC) and Sentinel-1 HyP3. Safe to re-run; the HyP3 part waits until jobs finish.

In [ ]:
import os, shutil, pandas as pd, boto3, hyp3_sdk, asf_search as asf
from botocore import UNSIGNED
from botocore.config import Config
from datetime import date

BASE = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/01_datos_crudos"
DOC = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/05_documentacion"
for d in ["sentinel1_hyp3", "nisar_gunw", "umbra_gec", "auxiliares"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
os.makedirs(DOC, exist_ok=True)
registro = []

# --- NISAR: los 6 GUNW cosísmicos (los 3 de la Celda 8 se copian, no se re-descargan) ---
nz = pd.read_csv("nisar_filtrado/nisar_productos.csv")
cos = nz[(nz.nivel == "GUNW") & (nz.tipo == "COSISMICO")]
for f in os.listdir("./datos_nisar_vargas") if os.path.isdir("./datos_nisar_vargas") else []:
    if f.endswith(".h5") and not os.path.exists(f"{BASE}/nisar_gunw/{f}"):
        shutil.copy(f"./datos_nisar_vargas/{f}", f"{BASE}/nisar_gunw/{f}")
pend = [u for u in cos.url if not os.path.exists(f"{BASE}/nisar_gunw/{os.path.basename(u)}")]
print(f"NISAR: {len(cos)} cosísmicos, faltan {len(pend)}")
if pend:
    asf.download_urls(urls=pend, path=f"{BASE}/nisar_gunw", session=session, processes=3)
registro += [("NISAR GUNW (ASF DAAC)", n) for n in cos.nombre]

# --- Umbra: 26 GEC ---
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED), region_name="us-west-2")
lst = pd.read_csv("umbra_inventario/umbra_listado_completo.csv")
gec = lst[lst.key.str.contains("GEC", case=False) & lst.key.str.lower().str.endswith((".tif", ".tiff"))]
print(f"Umbra: {len(gec)} GEC, {gec.bytes.sum()/1e9:.2f} GB")
for k in gec.key:
    dst = f"{BASE}/umbra_gec/{os.path.basename(k)}"
    if not os.path.exists(dst):
        s3.download_file("umbra-open-data-catalog", k, dst)
registro += [("Umbra GEC (s3://umbra-open-data-catalog)", os.path.basename(k)) for k in gec.key]

# --- Sentinel-1 HyP3: espera y descarga (puede tardar horas; se puede volver a correr) ---
hyp3 = hyp3_sdk.HyP3(username=usuario, password=clave)
for nombre in ["S1_025D_cosismico", "S1_033A_cosismico"]:
    jobs = hyp3.find_jobs(name=nombre)
    if len(jobs) == 0:
        print(f"[!] No existe el trabajo {nombre}: corra la Celda 10"); continue
    jobs = hyp3.watch(jobs)
    jobs.download_files(f"{BASE}/sentinel1_hyp3")
    registro.append(("Sentinel-1 HyP3 INSAR_GAMMA (ASF)", nombre))

pd.DataFrame(registro, columns=["fuente", "producto"]).assign(
    fecha_acceso=date.today().isoformat()).to_csv(f"{DOC}/registro_descargas.csv", index=False)
print("Listo. Registro de acceso -> 05_documentacion/registro_descargas.csv")


## Celda 12 · Preparar figuras de Drive para GitHub / *Cell 12 · Prepare Drive figures for GitHub*
🇪🇸 Busca las figuras preliminares en Drive, las reduce a ≤1600 px (JPG) y las deja en `para_github/docs/img/` con el nombre exacto que usa el README. Luego súbalas en GitHub: `docs/img` → *Add file → Upload files*.  
🇬🇧 Finds the preliminary figures in Drive, resizes them to ≤1600 px (JPG) and writes them to `para_github/docs/img/` with the exact names used by the README. Then upload them on GitHub: `docs/img` → *Add file → Upload files*.

In [ ]:
import glob, os
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
RAIZ_R = "/content/drive/My Drive/InSAR_LaGuaira_Resultados"
OUT = f"{RAIZ_R}/para_github/docs/img"
os.makedirs(OUT, exist_ok=True)
FIGURAS = {   # nombre en Drive -> nombre en GitHub (usado por el README)
    "nisar_mapa_cobertura.png": "prelim_cobertura_nisar_umbra.jpg",
    "umbra_mapa_adquisiciones.png": "prelim_mapa_umbra.jpg",
    "Sentinel1_Analisis_Suite.png": "prelim_s1_coherencia_fase_los.jpg",
    "Sentinel1_Composicion_Detallada.png": "prelim_s1_zooms_guaira.jpg",
    "Sentinel1_Correlacion_Dem.png": "prelim_s1_correlacion_dem.jpg",
    "Sentinel1_Monitoreo_Viaducto.png": "prelim_s1_viaducto.jpg",
}
for origen, destino in FIGURAS.items():
    hallados = glob.glob(f"{RAIZ_R}/**/{origen}", recursive=True)
    if not hallados:
        print("[!] no encontrado:", origen); continue
    im = Image.open(sorted(hallados)[-1]).convert("RGB")
    im.thumbnail((1600, 1600))
    im.save(f"{OUT}/{destino}", "JPEG", quality=85, optimize=True)
    print(f"✓ {destino}  {im.size}  {os.path.getsize(f'{OUT}/{destino}')/1e3:.0f} kB")
print("\nSuba estos archivos a GitHub en la carpeta docs/img / Upload these files to docs/img on GitHub:")
print(OUT)
